# Week 1 Day 5 Lab 2 assignment: where the harness is used, and a model comparison

The assignment, from the last text cell of `labs/week1day5_lab2.ipynb`: find where the agent uses its to-do list and its subagent, then compare other models on accuracy, cost and time.

The lab's own run (DeepSeek V4.1 Flash) is already in `labs/week1day5_lab2.ipynb`: 36 calls, $0.0416, 4/5 right. I added one run: `anthropic/claude-sonnet-5.5`, the model the assignment names for comparing with Lab 1 on the same model. Outputs are cleared; run it yourself to reproduce them, or read the findings directly in **My solution**.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you run this</h2>
            <span style="color:#EF476F;">This calls Claude Sonnet 5.5 through OpenRouter. The run below cost $0.82 for the five claims, more than DeepSeek or Lab 1's Sonnet run through the Claude Agent SDK; see the solution cell for why.
            You'll need <code>OPENROUTER_API_KEY</code> in your <code>.env</code> file.
            </span>
        </td>
    </tr>
</table>

In [ ]:
import json
import os
import shutil
import time
from pathlib import Path

import pandas as pd
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from dotenv import load_dotenv
from langchain.agents.middleware import TodoListMiddleware
from langchain_openai import ChatOpenAI


In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")


## The claims desk

Same five claims as the lab, built fresh into their own folder.

In [ ]:
DATA = Path("../data")
DESK = Path("lab2_desk_sonnet")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id."""
    claims = load("claims.json")
    decisions = load("evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels


def make_desk(desk, claims):
    """Build a fresh claims desk with the claims to decide and everything needed to decide them."""
    shutil.rmtree(desk, ignore_errors=True)
    for folder in ["claims", "policies", "customers", "decisions"]:
        (desk / folder).mkdir(parents=True)
    for claim in claims:
        (desk / "claims" / f"{claim['claim_id']}.json").write_text(json.dumps(claim, indent=2), encoding="utf-8")
    for policy in load("policies.json"):
        (desk / "policies" / f"{policy['policy_id']}.json").write_text(json.dumps(policy, indent=2), encoding="utf-8")
    for customer in load("customers.json"):
        (desk / "customers" / f"{customer['customer_id']}.json").write_text(json.dumps(customer, indent=2), encoding="utf-8")
    for name in ["policy_wording.md", "claims_manual.md", "company.json"]:
        shutil.copy(DATA / name, desk / name)


claims, labels = get_sample_data(tiny=True)
make_desk(DESK, claims)


## Where the to-do list and the subagent are used

`INSTRUCTIONS` tells the orchestrator to keep a to-do list (`write_todos`, from `TodoListMiddleware`) and hand each claim to the claim-handler subagent (`task`, naming the entry in `subagents=[claim_handler]` below). In a run these are the `write_todos: ...` lines and the `task: Decide claim CLM-xxxx` lines, with the subagent's own steps indented.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance, working through the queue of new claims on your own.

Everything you need is in this folder:
- claims/ has one file for each claim waiting in the queue
- policies/ and customers/ have the policy records and the customer records, one file each, named by their ids
- policy_wording.md is the policy wording and claims_manual.md is the claims handling manual
- company.json has the company's limits

Decide every claim in claims/. Keep a to-do list with one item for each claim, and give each claim to the claim-handler subagent, which decides it and writes its decision file.
When every claim has a decision, write handover.md at the top of this folder: a short note for the handler on what was decided and anything they should know.
"""

HANDLER_INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance. Decide the one claim you are given.

Read the claim in claims/, look up its policy in policies/ and its customer in customers/, and check policy_wording.md and claims_manual.md for the rules that apply.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, for one of the reasons the manual gives.

Write the decision to decisions/<claim id>.json with the fields claim_id, decision, payout (in dollars, 0 unless you approve) and rationale (2-3 sentences to explain the decision along with the relevant section of the wording or the manual).
Then reply with one line: the claim id, the decision and the payout.
"""


## The run: `anthropic/claude-sonnet-5.5`, in place of `deepseek/deepseek-v4.1-flash`

In [ ]:
llm = ChatOpenAI(
    model="anthropic/claude-sonnet-5.5",
    base_url=OPENROUTER_BASE_URL,
    api_key=openrouter_api_key,
    timeout=120,
    max_retries=5,
)

claim_handler = {
    "name": "claim-handler",
    "description": "Decides one claim and writes its decision file. Give it the claim id.",
    "system_prompt": HANDLER_INSTRUCTIONS,
}

agent = create_deep_agent(
    model=llm,
    system_prompt=INSTRUCTIONS,
    subagents=[claim_handler],
    middleware=[TodoListMiddleware()],
    backend=FilesystemBackend(root_dir=DESK, virtual_mode=True),
)


In [ ]:
def describe(tool_call):
    """A short line about one tool call."""
    args = tool_call["args"]
    if "file_path" in args or "path" in args:
        return f"{tool_call['name']} {args.get('file_path') or args.get('path')}"
    if tool_call["name"] == "task":
        return f"task: {args.get('description', '').splitlines()[0][:80]}"
    if tool_call["name"] == "write_todos":
        return "write_todos: " + ", ".join(f"{todo['content'][:30]} ({todo['status']})" for todo in args.get("todos", []))
    return tool_call["name"]


cost = 0
calls = 0
start = time.time()

for namespace, update in agent.stream(
    {"messages": [{"role": "user", "content": "Please work the queue."}]},
    stream_mode="updates",
    subgraphs=True,
    config={"recursion_limit": 200},
):
    if "model" in update:
        indent = "      " if namespace else ""
        for message in update["model"]["messages"]:
            calls += 1
            cost += message.response_metadata["token_usage"]["cost"]
            for tool_call in message.tool_calls:
                print(indent + describe(tool_call))
            if not namespace and message.text.strip():
                print(message.text.strip())

elapsed = time.time() - start
print(f"\nFinished after {calls} model calls for ${cost:.4f} in {elapsed:.0f} seconds")


## How it did

In [ ]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    payout = answer.get("payout", 0)
    if payout is None:
        return False
    return answer.get("decision") == label["decision"] and abs(float(payout) - label["payout"]) <= 1


rows = []
for claim_id, label in labels.items():
    path = DESK / "decisions" / f"{claim_id}.json"
    answer = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"decision": "missing", "payout": 0}
    rows.append({
        "claim": claim_id, "decision": answer.get("decision"), "payout": answer.get("payout"),
        "handler's decision": label["decision"], "handler's payout": label["payout"],
        "right": is_right(answer, label),
    })

results = pd.DataFrame(rows)
print(f"{results['right'].sum()} of {len(results)} right")
results


---

# My solution

**Where used.** The to-do list is `write_todos`; the subagent call is `task`, naming `subagents=[claim_handler]`.

**What I compared.** The lab's own DeepSeek run against one new run of `anthropic/claude-sonnet-5.5`, plus Lab 1's own Sonnet 5.5 run on the same five claims through the Claude Agent SDK instead of Deep Agents.

| Run | Harness | Right / 5 | Cost | Time |
|---|---|---|---|---|
| DeepSeek V4.1 Flash | Deep Agents | 4/5 | $0.04 | not timed |
| Sonnet 5.5 | Deep Agents | 5/5 | $0.82 | 113s |
| Sonnet 5.5 | Claude Agent SDK (Lab 1) | 5/5 | $0.35 | 44s |

Same model, same claims, same result, but Deep Agents cost more than twice as much and took more than twice as long. The difference is how each harness looks things up: Lab 1's claim helper reads one numbered section at a time (`list_sections`/`read_section`); Deep Agents only has raw file tools, so each subagent call re-reads `policy_wording.md` and `claims_manual.md` whole, for every claim. DeepSeek is cheap under the same harness only because the model itself is cheap per token, not because it reads less; it reads the same way.

On accuracy: Sonnet got all five right under both harnesses. DeepSeek was right on every decision but wrote a null payout instead of 0 on the referred claim, which the grading correctly counts as wrong. One run of each model does not make that a reliable difference, only one observed once.